# NeuroLens -- Kaggle Turnkey Runner

Runs the full pipeline end-to-end on a Kaggle Notebook GPU (T4 or P100,
16 GB VRAM), reading the CHB-MIT database from a Kaggle-hosted **Dataset**
(no external download):

0. Install dependencies + GPU/RAM sanity check
1. Locate + structurally validate the CHB-MIT dataset under `/kaggle/input`
   (checks the actual `chbXX-summary.txt` format and channel names match
   what `dataset.py` expects, in seconds, before committing hours of GPU
   time to a mismatched mirror)
2. Stage 0: synthetic Jansen-Rit ground-truth sanity check
3. `run_experiments.py` on chb01, chb02, chb03 (NeuroLens: Phase A/B training,
   FAISS export, full evaluation)
4. `run_ablations.py` (ResNet1D baseline + EMG/wander artifact-robustness
   comparison)
5. `generate_paper_artifacts.py` (all tables + all 5 figures, 300 DPI)
6. Bundle everything into `/kaggle/working/neurolens_paper_package.zip`

**Before running**: in Kaggle's right sidebar, add a CHB-MIT dataset as
input (see Section 1 below for which one and why), and turn on a GPU
accelerator under Settings -> Accelerator. Turn Internet **On** (needed for
`pip install` + `git clone` in Cells 1-2; this repo is public, no auth
needed).

**Resilience -- this notebook now survives a Kaggle session timeout**:
Kaggle GPU sessions have a hard per-session runtime cap (check
Settings -> your account's current quota; it has changed over Kaggle's
history and this notebook does not hardcode a number). A single LOSO run
across 17 folds x 2 models (NeuroLens + baseline) may not fit in one
session. As of this version, `train.py` and `run_ablations.py` **skip any
fold whose checkpoint already exists on disk** and write their summary
JSON after every fold (not just at the end) -- so if a session is killed
mid-run, re-running the *exact same* Cells 6/7 in a fresh session picks up
exactly where it left off instead of restarting from fold 1. See Section 5
("Chaining multiple sessions") for the save/reattach workflow this enables.
Pass `--force_retrain` (forwarded via `--extra_train_args` for Cell 6,
directly for Cell 7) if you deliberately want to redo everything.

Every stage's per-fold loop also has broad `except Exception` handling with
full-traceback logging and `continue` -- one anomalous fold (corrupt EDF, a
numerical singularity, a transient CUDA OOM) is logged and skipped, never
aborts the run.


In [ ]:
# Cell 1: Install dependencies
# Kaggle images ship torch already matched to the CUDA driver -- do NOT
# reinstall/pin torch here, that risks breaking the CUDA match.
!pip install -q mne scipy matplotlib scikit-learn umap-learn psutil

# faiss: try the modern CUDA-12 GPU wheel first, fall back to CPU if this
# image's CUDA version doesn't match or no GPU wheel is available. Whichever
# gets installed, xai_engine.py's TrajectoryVectorDB *also* falls back to
# CPU at runtime on its own if no GPU is visible, so either outcome here is
# safe.
!pip install -q faiss-gpu-cu12 || pip install -q faiss-cpu

import torch
print(f"torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}, {props.total_memory / 1e9:.1f} GB VRAM")
else:
    print("WARNING: no GPU visible -- check Settings > Accelerator is set to a GPU.")

import faiss
print(f"faiss {faiss.__version__}, GPU-capable build: {hasattr(faiss, 'StandardGpuResources')}")


In [ ]:
# Cell 2: Get the NeuroLens source and put it on sys.path
import os, sys

REPO_URL = "https://github.com/AbhayZ1/neurolens-eeg.git"
REPO_DIR = "/kaggle/working/neurolens-eeg"
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 $REPO_URL $REPO_DIR
    # If this is a private repo or you have no internet access enabled on
    # this kernel, instead add the repo as a Kaggle "Dataset" or "Utility
    # Script" attachment and point REPO_DIR at wherever it's mounted, e.g.
    # REPO_DIR = "/kaggle/input/neurolens-eeg-src/neurolens-eeg"

SRC_DIR = os.path.join(REPO_DIR, "src")
assert os.path.isdir(SRC_DIR), f"{SRC_DIR} not found -- see the fallback note above"
sys.path.insert(0, SRC_DIR)
print("SRC_DIR:", SRC_DIR)
print(sorted(os.listdir(SRC_DIR)))


In [ ]:
# Cell 3: Live resource diagnostics -- real numbers from *this* session,
# not assumed ones. Kaggle's RAM/disk/GPU allocation has changed over time
# and differs by accelerator choice, so read it directly instead of
# trusting a remembered figure.
import shutil
import psutil

def log_disk(label):
    for path in ["/kaggle/working", "/kaggle/input"]:
        if os.path.isdir(path):
            usage = shutil.disk_usage(path)
            print(f"[disk] {label}: {path} -> {usage.used/1e9:.2f} GB used / {usage.total/1e9:.2f} GB total "
                  f"({usage.free/1e9:.2f} GB free)")

def log_ram(label):
    vm = psutil.virtual_memory()
    print(f"[ram]  {label}: {vm.used/1e9:.2f} GB used / {vm.total/1e9:.2f} GB total "
          f"({vm.available/1e9:.2f} GB available)")

def log_gpu_mem(label):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e6
        reserved = torch.cuda.memory_reserved() / 1e6
        total = torch.cuda.get_device_properties(0).total_memory / 1e6
        print(f"[gpu]  {label}: allocated={alloc:.1f} MB reserved={reserved:.1f} MB / {total:.1f} MB total")

log_disk("startup")
log_ram("startup")
log_gpu_mem("startup")

print()
print("If [ram] 'available' here is much lower than you expect, close/restart other")
print("sessions -- Kaggle RAM is per-account-session-family in some tiers, not just")
print("per-notebook. If [disk] free is below ~15 GB, the FAISS index export across")
print("17 folds (Cell 6) may not fit; see the disk troubleshooting note at the bottom.")


## 1. Add + validate the CHB-MIT dataset

**Add this as a Kaggle Input** (right sidebar -> Add Input -> search):
[`CHB-MIT Scalp EEG Database Annotated`](https://www.kaggle.com/datasets/minhtranv/chb-mit-scalp-eeg-database-annotated)
-- its description text matches PhysioNet's own documentation verbatim
(23 cases, continuous `.edf` files, `chbXX-summary.txt` per patient).

**This has not been independently verified against `dataset.py`'s exact
parsing requirements** -- Kaggle's dataset file browser can't be inspected
without actually mounting it in a session. Cell 4 below is a fast (~seconds)
structural preflight that checks the *actual* mounted files against exactly
what `dataset.py` needs, **before** Cell 6 commits hours of GPU time:

- `chbXX-summary.txt` exists and parses (`File Name:` / `File Start Time:` /
  `Seizure Start Time: N seconds` -- the exact regex `dataset.py` uses)
- the `.edf` files it lists are actually present
- the 18-channel bipolar montage (`FP1-F7`, `F7-T7`, ... `CZ-PZ`) is present
  as named channels in the first `.edf`

If this fails, the dataset doesn't match -- try one of these alternates
instead (swap the search term in "Add Input"), then re-run Cell 4:
- `chb-mit-scalp-eeg-database-1.0.0-filtred-labled` (imenej)
- Do **not** use anything named "seizure only" or a `.csv` conversion --
  those don't have the continuous recordings `dataset.py` needs to compute
  interictal windows, or aren't in `.edf` format at all.


In [ ]:
# Cell 4: Resolve + structurally validate the dataset for each patient
# BEFORE committing GPU time to Cell 6. Fails fast and loud (not hours in)
# if the mounted Kaggle dataset doesn't actually match what dataset.py needs.
import re
from dataset import resolve_data_dir, parse_summary_file, BIPOLAR_MONTAGE
import mne
mne.set_log_level("ERROR")

DATA_DIR = "/kaggle/input"  # broad root; narrow this if you know the exact path
PATIENTS = ["chb01", "chb02", "chb03"]

all_ok = True
for p in PATIENTS:
    try:
        resolved = resolve_data_dir(DATA_DIR, p)
    except FileNotFoundError as exc:
        print(f"{p}: NOT FOUND -- {exc}")
        all_ok = False
        continue

    summary_path = os.path.join(resolved, p, f"{p}-summary.txt")
    try:
        entries = parse_summary_file(summary_path)
    except Exception as exc:
        print(f"{p}: FOUND but summary.txt failed to parse -- {exc}")
        all_ok = False
        continue
    if not entries or not any(e.seizures_sec for e in entries) and p != "chb01":
        # not fatal by itself (some patients legitimately have sparse seizures
        # in a subset), just informational
        pass

    first_edf = os.path.join(resolved, p, entries[0].filename)
    if not os.path.isfile(first_edf):
        print(f"{p}: summary.txt parsed ({len(entries)} file entries) but "
              f"{entries[0].filename} is missing on disk -- mirror is incomplete")
        all_ok = False
        continue

    try:
        raw = mne.io.read_raw_edf(first_edf, preload=False, verbose=False)
        available = {ch.strip().upper() for ch in raw.ch_names}
        missing = [ch for ch in BIPOLAR_MONTAGE if ch.upper() not in available]
    except Exception as exc:
        print(f"{p}: could not read {first_edf} with MNE -- {exc}")
        all_ok = False
        continue

    if missing:
        print(f"{p}: FOUND ({len(entries)} files, sfreq={raw.info['sfreq']:.0f} Hz) "
              f"but missing required channel(s): {missing}")
        all_ok = False
    else:
        n_seizures = sum(len(e.seizures_sec) for e in entries)
        print(f"{p}: OK -- {resolved} ({len(entries)} files, {n_seizures} seizures, "
              f"sfreq={raw.info['sfreq']:.0f} Hz, all {len(BIPOLAR_MONTAGE)} channels present)")

print()
if all_ok:
    print("All patients validated. Safe to proceed to Cell 5+.")
else:
    print("VALIDATION FAILED for at least one patient -- do not run Cell 6 yet. "
          "Swap the Kaggle Input dataset (see the markdown above) and re-run this cell.")


## 2. Stage 0 -- Synthetic Jansen-Rit ground-truth sanity check

In [ ]:
# Cell 5: Stage 0 (also runs automatically as part of run_experiments.py,
# but running it standalone first fails fast in ~1 minute if something in
# the JR-NMM integrator is broken, before committing to a multi-hour job).
from synthetic_nmm import JansenRitParams, generate_synthetic_bifurcation_dataset
import numpy as np

synth = generate_synthetic_bifurcation_dataset(
    duration_mins=15.0, sampling_rate=256, n_channels=18,
    transition_start_min=10.0, transition_duration_min=0.5,
    params=JansenRitParams(), seed=1000,
)
eeg = synth.eeg_uv.numpy()
assert np.isfinite(eeg).all(), "synthetic EEG contains non-finite values"
print(f"OK: eeg_uv shape={eeg.shape}, p_crit={synth.ground_truth.p_crit:.3f}, "
      f"bifurcation onset={synth.ground_truth.bifurcation_onset_sec:.3f}s")


## 3. Run the pipeline

Hyperparameters below are chosen to comfortably fit a 16 GB GPU (T4 or
P100) for 3 patients x their LOSO folds (chb01: 7, chb02: 3, chb03: 7 ->
17 folds total for NeuroLens, then 17 more for the baseline). Scale
`--epochs_pretrain` / `--epochs_finetune` / `--mc_samples` up if you have
time to spare, or set `--max_folds_per_patient` to a small number for a
quick smoke test of the whole pipeline before committing to the full run.

**Note on multi-GPU accelerator options** (e.g. "GPU T4 x2"): this
pipeline has no multi-GPU/`DataParallel` code -- it only ever uses one
GPU regardless of how many are attached. A single T4/P100 is functionally
equivalent to picking the x2 option; the second GPU just sits idle.

**If this is fold 1 of a fresh run**, budget your time first: after the
first 1-2 folds finish in the log, `experiment_run.log` shows a real
per-fold duration on *your* actual hardware -- multiply by 17 to estimate
whether the full NeuroLens stage fits in one session. If it clearly won't,
stop the session deliberately (Save Version) rather than let it get killed
mid-fold, then resume per Section 5 below -- the folds already checkpointed
are preserved either way, since `--force_retrain` isn't passed.


In [ ]:
# Cell 6: run_experiments.py -- NeuroLens training + evaluation
# Add --extra_train_args "--force_retrain" below to force-redo every fold
# instead of resuming from existing checkpoints.
OUTPUT_DIR = "/kaggle/working/runs/exp1"

!python {SRC_DIR}/run_experiments.py \
    --data_dir {DATA_DIR} \
    --output_dir {OUTPUT_DIR} \
    --patients {' '.join(PATIENTS)} \
    --device cuda \
    --epochs_pretrain 10 --epochs_finetune 15 \
    --batch_size 32 --num_workers 2 \
    --mc_samples 30 \
    --synthetic_duration_min 15 --synthetic_transition_start_min 10 --synthetic_n_trials 3

log_disk("after run_experiments.py")
log_ram("after run_experiments.py")
log_gpu_mem("after run_experiments.py")


In [ ]:
# Cell 7: run_ablations.py -- ResNet1D baseline + EMG/wander robustness comparison
# Add --force_retrain below to force-redo every baseline fold instead of
# resuming from existing checkpoints.
!python {SRC_DIR}/run_ablations.py \
    --data_dir {DATA_DIR} \
    --output_dir {OUTPUT_DIR} \
    --patients {' '.join(PATIENTS)} \
    --device cuda \
    --epochs_baseline 15 --lr_baseline 1e-3 \
    --batch_size 32 --num_workers 2 \
    --worst_case_severity 4.0 \
    --robustness_mc_samples 20

log_disk("after run_ablations.py")
log_ram("after run_ablations.py")
log_gpu_mem("after run_ablations.py")


In [ ]:
# Cell 8: generate_paper_artifacts.py -- all tables + all 5 figures at 300 DPI
CKPT_DIR = f"{OUTPUT_DIR}/checkpoints"
BASELINE_DIR = f"{OUTPUT_DIR}/baseline_checkpoints"
ABLATION_DIR = f"{OUTPUT_DIR}/ablation_artifacts"
PAPER_DIR = f"{OUTPUT_DIR}/paper_artifacts"

!python {SRC_DIR}/generate_paper_artifacts.py \
    --report_json {CKPT_DIR}/evaluation_report.json \
    --baseline_report_json {BASELINE_DIR}/baseline_evaluation_report.json \
    --robustness_json {ABLATION_DIR}/robustness_comparison.json \
    --output_dir {PAPER_DIR}

log_disk("after generate_paper_artifacts.py")

import json
with open(f"{PAPER_DIR}/artifact_generation_summary.json") as f:
    print(json.dumps(json.load(f), indent=2))


## 4. Bundle everything into one downloadable package

In [ ]:
# Cell 9: zip all .tex / .png / .json artifacts into one package
import zipfile
import glob

ZIP_PATH = "/kaggle/working/neurolens_paper_package.zip"
patterns = ["**/*.tex", "**/*.png", "**/*.json", "**/*.log"]

files_to_zip = []
for pattern in patterns:
    files_to_zip.extend(glob.glob(os.path.join(OUTPUT_DIR, pattern), recursive=True))
files_to_zip = sorted(set(files_to_zip))

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in files_to_zip:
        arcname = os.path.relpath(path, OUTPUT_DIR)
        zf.write(path, arcname)

print(f"Zipped {len(files_to_zip)} file(s) into {ZIP_PATH}")
print(f"Package size: {os.path.getsize(ZIP_PATH) / 1e6:.2f} MB")
log_disk("final")


## 5. Chaining multiple sessions (if 17+17 folds don't fit in one)

Kaggle GPU sessions have a hard runtime cap that has changed over time --
check your account's current limit under Settings rather than trusting a
remembered number. If Cell 6 or 7 gets killed by that cap partway through:

1. **Before the session ends** (or immediately after Kaggle kills it):
   click **Save Version** -> **Save & Run All** (or just **Save**) so
   `/kaggle/working/runs/exp1` -- including every fold checkpointed so far
   -- is persisted as this notebook's Output.
2. **Start a new session** on this same notebook (or open the saved
   version). `/kaggle/working` resets empty each session, so:
   - Add this notebook's own previous **Output** as a new Input (sidebar ->
     Add Input -> Notebook Output -> this notebook's last saved version).
   - Add a cell before Cell 6 that copies it back into place:
     ```python
     import shutil
     prior = "/kaggle/input/<your-notebook-slug>/runs/exp1"  # adjust to the actual mounted path
     if os.path.isdir(prior) and not os.path.isdir(OUTPUT_DIR):
         shutil.copytree(prior, OUTPUT_DIR)
         print(f"Restored prior progress from {prior}")
     ```
3. **Re-run Cells 6/7 unchanged.** Because neither `--force_retrain` flag is
   passed, every fold whose checkpoint already exists is skipped (logged as
   `checkpoint already exists, skipping`) and only the remaining folds
   train -- `training_summary.json` / `baseline_training_summary.json` are
   rewritten after every fold, so this is safe no matter how many times you
   repeat it.
4. Once all 17+17 folds show up in the logs, continue to Cell 8 (artifacts)
   and Cell 9 (zip) as normal.


## Troubleshooting

- **Cell 4 validation fails**: see the markdown in Section 1 -- try a
  different Kaggle dataset for the CHB-MIT input. Don't proceed to Cell 6
  until Cell 4 prints `All patients validated.`
- **"No trajectories of length k could be built" / a fold is skipped
  entirely**: that fold's training set had too little data surviving LOSO
  exclusion. Check the fold's line in `run_experiments.log` /
  `ablation_run.log`; the run continues regardless.
- **CUDA OOM**: lower `--batch_size` and/or `--mc_samples` in Cells 6-7 and
  re-run just that cell -- checkpoints already written for earlier folds
  are untouched, and (as of this version) will be skipped rather than
  retrained when you re-run.
- **Session killed by Kaggle's runtime cap partway through Cell 6 or 7**:
  not a failure -- see Section 5 above ("Chaining multiple sessions").
  Every fold checkpointed before the kill is safe; the summary JSON is
  rewritten after each fold specifically so this can't lose progress.
- **Fig 3 (ROC/PR) or Fig 1 (latent map) missing from the zip**: printed as
  `SKIPPED` with a reason in Cell 8's output -- usually a fold's pooled
  test predictions ended up single-class (needs both classes to draw an
  ROC/PR curve) or a fold has no exported FAISS index. Every other artifact
  is still produced; this is graceful degradation, not a crash.
- **Disk or RAM near the limit**: `log_disk(...)` / `log_ram(...)` after
  each stage (and at startup, Cell 3) show exactly where usage stands.
  Checkpoints never include optimizer state (see `train.py` /
  `run_ablations.py`); the FAISS trajectory export is the usual largest
  disk consumer -- reduce it by increasing `--trajectory_stride` when
  calling `run_experiments.py` (forwarded to `train.py`).
